# APIP Research Engine v1.0

Canonical research reference implementation for the APIP Trading Intelligence & Performance Platform.

This notebook consolidates the previous research notebook chain into one deterministic, modular notebook and mirrors the production service architecture in `APIP_NEXTJS_PLATFORM_SPEC_V1_6`.

## Core rules
- Opportunity is the central object.
- Shadow trades are internal only.
- Analysts see coaching ranges, trigger probability, expected R, event risk and condition warnings.
- Historical backfill rows can support raw outcome profiling but cannot be used for coaching-alignment reviews.
- Every recommendation should be reproducible from its parameter snapshot.

## Notebook service map

| Notebook Section | Production Service |
|---|---|
| Market State | MarketStateService |
| Market Regime | RegimeService |
| Economic Calendar | EconomicCalendarService |
| Template Engine | TemplateService |
| Analyst Profile | AnalystProfileService |
| Entry Optimiser | EntryOptimizerService |
| Trigger Probability | TriggerProbabilityService |
| Expected R | ExpectedRService |
| Condition Awareness | RecommendationValidityService |
| Recommendation Version | RecommendationVersionService |
| Shadow Engine | ShadowTradeService |
| Coaching Engine | CoachingService |
| Review Engine | ReviewService |
| Allocation Engine | AllocationService |

In [ ]:
# ============================================================
# 1. CONFIGURATION
# ============================================================
import os, math, json, uuid, hashlib, warnings
from datetime import datetime, timedelta
from typing import Optional, Dict, Any, List, Tuple
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.utcnow().isoformat()

PARAMETERS = {
    'atr_period': 14,
    'zone_count': 4,
    'stale_atr_threshold': 0.25,
    'force_recalc_atr_threshold': 0.50,
    'minimum_rr': 2.0,
    'min_trigger_sample': 20,
    'fallback_trigger_probability': 0.50,
    'min_template_trades': 10,
    'min_profile_trades': 10,
    'high_confidence_min_trades': 50,
    'medium_confidence_min_trades': 20,
    'sessions': {
        'Europe': {'engine_run_window_uk': '05:45-06:00', 'publication_window_uk': '06:00-07:00', 'expiry_rule': 'same_day_21:00_UK'},
        'US': {'engine_run_window_uk': '11:45-12:00', 'publication_window_uk': '12:00-14:00', 'expiry_rule': 'same_day_21:00_UK'},
        'APAC': {'engine_run_window_uk': '15:45-16:00', 'publication_window_uk': '16:00-18:00', 'expiry_rule': 'next_day_16:00_UK'},
        'Crypto': {'engine_run_window_uk': 'flexible', 'publication_window_uk': 'continuous', 'expiry_rule': 'next_day_12:00_UK'},
    },
    'forbidden_analyst_terms': ['shadow trade', 'model beat you', 'non-compliant', 'wrong', 'failure', 'low confidence']
}

def stable_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode('utf-8')).hexdigest()

PARAMETER_SNAPSHOT = json.loads(json.dumps(PARAMETERS))
PARAMETER_SNAPSHOT_HASH = stable_hash(PARAMETER_SNAPSHOT)
print('APIP Research Engine v1.0')
print('Run ID:', RUN_ID)
print('Parameter snapshot hash:', PARAMETER_SNAPSHOT_HASH)

In [ ]:
# ============================================================
# 1A. MARKET UNIVERSE + ANALYST CONSTRAINTS
# ============================================================
MARKET_UNIVERSE = pd.DataFrame([
    {'market': 'EURUSD', 'asset_class': 'FX', 'session': 'Europe', 'finnhub_symbol': 'OANDA:EUR_USD'},
    {'market': 'GBPUSD', 'asset_class': 'FX', 'session': 'Europe', 'finnhub_symbol': 'OANDA:GBP_USD'},
    {'market': 'USDJPY', 'asset_class': 'FX', 'session': 'APAC', 'finnhub_symbol': 'OANDA:USD_JPY'},
    {'market': 'Gold', 'asset_class': 'Metal', 'session': 'US', 'finnhub_symbol': 'OANDA:XAU_USD'},
    {'market': 'SP500', 'asset_class': 'Index', 'session': 'US', 'finnhub_symbol': 'OANDA:SPX500_USD'},
    {'market': 'Bitcoin', 'asset_class': 'Crypto', 'session': 'Crypto', 'finnhub_symbol': 'BINANCE:BTCUSDT'},
])

ACTIVE_ANALYSTS = pd.DataFrame([
    {'analyst': 'IAN', 'name': 'Ian', 'active': True, 'Europe': True, 'US': True, 'APAC': False},
    {'analyst': 'TIV', 'name': 'Tibor', 'active': True, 'Europe': True, 'US': True, 'APAC': True},
    {'analyst': 'MOH', 'name': 'Mona', 'active': True, 'Europe': True, 'US': True, 'APAC': True},
    {'analyst': 'MAG', 'name': 'Maged', 'active': True, 'Europe': True, 'US': False, 'APAC': True},
    {'analyst': 'KG',  'name': 'Khaled', 'active': True, 'Europe': True, 'US': False, 'APAC': True},
])

display(MARKET_UNIVERSE)
display(ACTIVE_ANALYSTS)

# 2. Data contracts

Expected logical inputs:

## `historical_trades`
Required: `date`, `market`, `analyst`, `direction`, `entry`, `stop`, `target`, `result_r`, `triggered`, `session`, `historical_backfill`.

## `ohlc_data`
Required: `market`, `date`, `open`, `high`, `low`, `close`.

## `current_prices`
Required: `market`, `captured_at`, `current_price`.

## `economic_events`
Required: `event_time_uk`, `currency`, `event_name`, `impact`.

In [ ]:
# ============================================================
# 2A. LOAD DATA OR GENERATE SAMPLE DATA
# ============================================================
def require_columns(df, required, name):
    missing = [c for c in required if c not in df.columns]
    if missing: raise ValueError(f'{name} missing required columns: {missing}')

def ensure_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns: out[col] = pd.to_datetime(out[col], errors='coerce')
    return out

def normalise_direction(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().upper()
    if s in ['BUY','LONG','BULLISH']: return 'BUY'
    if s in ['SELL','SHORT','BEARISH']: return 'SELL'
    return s

PATHS = {'historical_trades': None, 'ohlc_data': None, 'current_prices': None, 'economic_events': None}

def load_csv_or_empty(path, columns):
    return pd.read_csv(path) if path and os.path.exists(path) else pd.DataFrame(columns=columns)

historical_trades = load_csv_or_empty(PATHS['historical_trades'], ['date','market','analyst','direction','entry','stop','target','result_r','triggered','session','historical_backfill'])
ohlc_data = load_csv_or_empty(PATHS['ohlc_data'], ['market','date','open','high','low','close'])
current_prices = load_csv_or_empty(PATHS['current_prices'], ['market','captured_at','current_price'])
economic_events = load_csv_or_empty(PATHS['economic_events'], ['event_time_uk','currency','event_name','impact'])

def generate_sample_data(markets=MARKET_UNIVERSE['market'].tolist(), days=140, seed=42):
    rng = np.random.default_rng(seed)
    start = pd.Timestamp.today().normalize() - pd.Timedelta(days=days)
    price_seed = {'EURUSD':1.08,'GBPUSD':1.27,'USDJPY':155.0,'Gold':2350.0,'SP500':5300.0,'Bitcoin':65000.0}
    rows, trade_rows = [], []
    for m in markets:
        price = price_seed.get(m, 100.0); vol = max(price*0.006, 0.001)
        for i in range(days):
            d = start + pd.Timedelta(days=i)
            if d.weekday() >= 5 and m != 'Bitcoin': continue
            ret = rng.normal(0, vol); o = price; c = max(0.0001, price + ret)
            h = max(o,c) + abs(rng.normal(0, vol*0.7)); l = min(o,c) - abs(rng.normal(0, vol*0.7))
            rows.append({'market':m,'date':d,'open':o,'high':h,'low':l,'close':c}); price = c
            if rng.random() < 0.16:
                analyst = rng.choice(ACTIVE_ANALYSTS['analyst'].tolist()); direction = rng.choice(['BUY','SELL'])
                result_r = rng.choice([-1,-0.5,0,1,2,3], p=[.30,.10,.10,.25,.18,.07])
                trade_rows.append({'date':d,'market':m,'analyst':analyst,'direction':direction,'entry':c,'stop':c-vol if direction=='BUY' else c+vol,'target':c+2*vol if direction=='BUY' else c-2*vol,'result_r':result_r,'triggered':True,'session':MARKET_UNIVERSE.set_index('market').loc[m,'session'],'historical_backfill':True,'source_system':'SAMPLE','source_record_id':str(uuid.uuid4())})
    ohlc = pd.DataFrame(rows); trades = pd.DataFrame(trade_rows)
    current = ohlc.sort_values('date').groupby('market').tail(1)[['market','close']].rename(columns={'close':'current_price'})
    current['captured_at'] = pd.Timestamp.utcnow()
    events = pd.DataFrame([{'event_time_uk':pd.Timestamp.utcnow()+pd.Timedelta(hours=2),'currency':'USD','event_name':'US CPI','impact':'HIGH'},{'event_time_uk':pd.Timestamp.utcnow()+pd.Timedelta(hours=6),'currency':'GBP','event_name':'BoE Speech','impact':'MEDIUM'}])
    return trades, ohlc, current, events

if historical_trades.empty or ohlc_data.empty or current_prices.empty:
    historical_trades, ohlc_data, current_prices, economic_events = generate_sample_data()

historical_trades = ensure_datetime(historical_trades, ['date'])
ohlc_data = ensure_datetime(ohlc_data, ['date'])
current_prices = ensure_datetime(current_prices, ['captured_at'])
economic_events = ensure_datetime(economic_events, ['event_time_uk'])
historical_trades['direction'] = historical_trades['direction'].apply(normalise_direction)
print('Data shapes:', historical_trades.shape, ohlc_data.shape, current_prices.shape, economic_events.shape)

In [ ]:
# ============================================================
# 3. MARKET STATE ENGINE
# ============================================================
def calculate_atr(df, period=14):
    require_columns(df, ['market','date','high','low','close'], 'ohlc_data')
    out = df.sort_values(['market','date']).copy()
    out['prev_close'] = out.groupby('market')['close'].shift(1)
    out['tr1'] = out['high'] - out['low']
    out['tr2'] = (out['high'] - out['prev_close']).abs()
    out['tr3'] = (out['low'] - out['prev_close']).abs()
    out['true_range'] = out[['tr1','tr2','tr3']].max(axis=1)
    out['atr14'] = out.groupby('market')['true_range'].transform(lambda s: s.rolling(period, min_periods=period).mean())
    return out.drop(columns=['tr1','tr2','tr3'], errors='ignore')

def calculate_atr_zones(row, current_price=None, zone_count=4):
    high, low, atr, close = row.get('high',np.nan), row.get('low',np.nan), row.get('atr14',np.nan), row.get('close',np.nan)
    price = current_price if current_price is not None and not pd.isna(current_price) else close
    if pd.isna(high) or pd.isna(low) or pd.isna(atr) or atr <= 0:
        return {'lower_band':np.nan,'zone1_top':np.nan,'zone2_top':np.nan,'zone3_top':np.nan,'upper_band':np.nan,'current_zone':np.nan}
    lower_band = high - atr; upper_band = low + atr
    if upper_band <= lower_band:
        lower_band, upper_band = close - atr/2, close + atr/2
    step = (upper_band - lower_band) / zone_count
    z1, z2, z3 = lower_band + step, lower_band + 2*step, lower_band + 3*step
    if price < lower_band: cz = 'Too Deep'
    elif price <= z1: cz = 'Zone 1'
    elif price <= z2: cz = 'Zone 2'
    elif price <= z3: cz = 'Zone 3'
    elif price <= upper_band: cz = 'Zone 4'
    else: cz = 'Too High'
    return {'lower_band':lower_band,'zone1_top':z1,'zone2_top':z2,'zone3_top':z3,'upper_band':upper_band,'current_zone':cz}

def build_market_state(ohlc, current, parameters=PARAMETERS):
    atr_df = calculate_atr(ohlc, parameters['atr_period'])
    latest = atr_df.sort_values('date').groupby('market').tail(1).copy()
    current_latest = current.sort_values('captured_at').groupby('market').tail(1)
    latest = latest.merge(current_latest[['market','captured_at','current_price']], on='market', how='left')
    zones = pd.DataFrame([calculate_atr_zones(row, row.get('current_price'), parameters['zone_count']) for _, row in latest.iterrows()])
    out = pd.concat([latest.reset_index(drop=True), zones.reset_index(drop=True)], axis=1)
    out['state_generated_at'] = pd.Timestamp.utcnow(); out['parameter_snapshot_hash'] = PARAMETER_SNAPSHOT_HASH
    return out
market_state = build_market_state(ohlc_data, current_prices)
display(market_state.head())

In [ ]:
# ============================================================
# 4. MARKET REGIME + EVENT RISK ENGINE
# ============================================================
def derive_market_regime(ohlc):
    df = ohlc.sort_values(['market','date']).copy()
    df['ema20'] = df.groupby('market')['close'].transform(lambda s: s.ewm(span=20, adjust=False).mean())
    df['ema50'] = df.groupby('market')['close'].transform(lambda s: s.ewm(span=50, adjust=False).mean())
    df['ema20_slope'] = df.groupby('market')['ema20'].diff(5)
    df['return_abs'] = df.groupby('market')['close'].pct_change().abs()
    df['vol20'] = df.groupby('market')['return_abs'].transform(lambda s: s.rolling(20, min_periods=10).mean())
    df['vol60'] = df.groupby('market')['return_abs'].transform(lambda s: s.rolling(60, min_periods=20).mean())
    latest = df.groupby('market').tail(1)[['market','ema20','ema50','ema20_slope','vol20','vol60']].copy()
    latest['trend_state'] = latest.apply(lambda r: 'TRENDING_UP' if r.ema20>r.ema50 and r.ema20_slope>0 else ('TRENDING_DOWN' if r.ema20<r.ema50 and r.ema20_slope<0 else 'RANGE'), axis=1)
    latest['volatility_state'] = latest.apply(lambda r: 'UNKNOWN' if pd.isna(r.vol20) or pd.isna(r.vol60) or r.vol60==0 else ('HIGH_VOL' if r.vol20/r.vol60>=1.5 else ('LOW_VOL' if r.vol20/r.vol60<=0.7 else 'NORMAL_VOL')), axis=1)
    latest['regime_tags'] = latest.apply(lambda r: [r.trend_state.lower(), r.volatility_state.lower()], axis=1)
    latest['regime_confidence'] = np.where(latest['volatility_state'].eq('UNKNOWN'), 'LOW', 'MEDIUM')
    latest['captured_at'] = pd.Timestamp.utcnow()
    return latest

CURRENCY_MARKET_MAP = {'USD':['EURUSD','GBPUSD','USDJPY','Gold','SP500','Bitcoin'],'EUR':['EURUSD'],'GBP':['GBPUSD'],'JPY':['USDJPY']}
def map_event_risk(events, now=None):
    now = pd.Timestamp.utcnow() if now is None else now
    if events.empty: return pd.DataFrame(columns=['market','event_name','currency','impact','event_time_uk','event_risk_status','risk_score','analyst_warning'])
    rows=[]; ev=events.copy(); ev['event_time_uk']=pd.to_datetime(ev['event_time_uk'], errors='coerce')
    for _, e in ev.iterrows():
        currency=str(e.get('currency','')).upper(); impact=str(e.get('impact','')).upper(); affected=CURRENCY_MARKET_MAP.get(currency, [])
        hrs=(e['event_time_uk']-now).total_seconds()/3600 if pd.notna(e['event_time_uk']) else np.nan
        for market in affected:
            if impact=='HIGH' and -1 <= hrs <= 3: status='EVENT_ACTIVE' if hrs<=0 else 'HIGH_RISK'; score=.9
            elif impact in ['HIGH','MEDIUM'] and 0 < hrs <= 8: status='WATCH'; score=.5 if impact=='MEDIUM' else .7
            else: status='NONE'; score=0
            warning = f'{impact.title()} impact {currency} event: {e.get("event_name")} at {e.get("event_time_uk")}' if status!='NONE' else ''
            rows.append({'market':market,'event_name':e.get('event_name'),'currency':currency,'impact':impact,'event_time_uk':e.get('event_time_uk'),'event_risk_status':status,'risk_score':score,'analyst_warning':warning})
    return pd.DataFrame(rows)
market_regime = derive_market_regime(ohlc_data)
event_risk = map_event_risk(economic_events)
display(market_regime.head()); display(event_risk.head())

In [ ]:
# ============================================================
# 5. TEMPLATE + ANALYST PROFILE ENGINES
# ============================================================
def add_entry_zone_if_missing(trades):
    out = trades.copy()
    if 'entry_zone' not in out.columns:
        out['entry_zone'] = out['atr_zone'] if 'atr_zone' in out.columns else np.nan
    return out

def build_template_profiles(trades):
    if trades.empty: return pd.DataFrame(columns=['market','direction','entry_zone','trades','avg_r','win_rate','trigger_rate','template_quality'])
    t = add_entry_zone_if_missing(trades); t['win'] = pd.to_numeric(t['result_r'], errors='coerce') > 0
    t['triggered_num'] = t['triggered'].astype(bool).astype(int) if 'triggered' in t.columns else 1
    prof = t.groupby(['market','direction','entry_zone'], dropna=False).agg(trades=('result_r','size'), avg_r=('result_r','mean'), win_rate=('win','mean'), trigger_rate=('triggered_num','mean')).reset_index()
    prof['template_quality'] = np.select([(prof.trades>=PARAMETERS['high_confidence_min_trades'])&(prof.avg_r>0),(prof.trades>=PARAMETERS['medium_confidence_min_trades'])&(prof.avg_r>0)], ['HIGH','MEDIUM'], default='LOW')
    return prof.sort_values(['market','avg_r','trades'], ascending=[True,False,False])

def build_analyst_profiles(trades):
    if trades.empty: return pd.DataFrame(columns=['analyst','market','direction','entry_zone','trades','avg_r','win_rate','profile_quality'])
    t = add_entry_zone_if_missing(trades); t['win'] = pd.to_numeric(t['result_r'], errors='coerce') > 0
    prof = t.groupby(['analyst','market','direction','entry_zone'], dropna=False).agg(trades=('result_r','size'), avg_r=('result_r','mean'), win_rate=('win','mean')).reset_index()
    prof['profile_quality'] = np.select([(prof.trades>=PARAMETERS['high_confidence_min_trades'])&(prof.avg_r>0),(prof.trades>=PARAMETERS['medium_confidence_min_trades'])&(prof.avg_r>0)], ['HIGH','MEDIUM'], default='LOW')
    return prof.sort_values(['market','analyst','avg_r'], ascending=[True,True,False])

template_profiles = build_template_profiles(historical_trades)
analyst_profiles = build_analyst_profiles(historical_trades)
display(template_profiles.head()); display(analyst_profiles.head())

In [ ]:
# ============================================================
# 6. SELECTION, ENTRY, TRIGGER AND EXPECTED R
# ============================================================
def select_best_template(market, templates):
    subset = templates[templates.market.eq(market)].copy()
    subset = subset[pd.to_numeric(subset.trades, errors='coerce') >= PARAMETERS['min_template_trades']]
    if subset.empty:
        return {'template_source':'fallback','direction':'BUY','preferred_entry_zone':'Zone 1','template_avg_r':0.0,'template_win_rate':np.nan,'template_trades':0,'template_quality':'LOW'}
    best = subset.sort_values(['avg_r','trades','win_rate'], ascending=[False,False,False]).iloc[0]
    return {'template_source':'historical_template','direction':best.direction,'preferred_entry_zone':best.entry_zone if pd.notna(best.entry_zone) else 'Zone 1','template_avg_r':best.avg_r,'template_win_rate':best.win_rate,'template_trades':int(best.trades),'template_quality':best.template_quality}

def select_best_analyst(market, direction, zone, profiles, active_analysts, session):
    eligible = active_analysts[(active_analysts.active) & (active_analysts.get(session, False)==True)]['analyst'].tolist()
    subset = profiles[(profiles.market.eq(market)) & (profiles.analyst.isin(eligible))].copy()
    exact = subset[(subset.direction.eq(direction)) & (subset.entry_zone.eq(zone))]
    if not exact.empty: best = exact.sort_values(['avg_r','trades','win_rate'], ascending=[False,False,False]).iloc[0]; source='exact_profile'
    elif not subset.empty: best = subset.sort_values(['avg_r','trades','win_rate'], ascending=[False,False,False]).iloc[0]; source='market_profile'
    else: return {'assigned_analyst': eligible[0] if eligible else None, 'profile_source':'fallback','profile_avg_r':0.0,'profile_win_rate':np.nan,'profile_trades':0,'profile_quality':'LOW','eligible_analysts':eligible}
    return {'assigned_analyst':best.analyst,'profile_source':source,'profile_avg_r':best.avg_r,'profile_win_rate':best.win_rate,'profile_trades':int(best.trades),'profile_quality':best.profile_quality,'eligible_analysts':eligible}

def zone_bounds(row, zone):
    mapping = {'Zone 1':(row.lower_band,row.zone1_top),'Zone 2':(row.zone1_top,row.zone2_top),'Zone 3':(row.zone2_top,row.zone3_top),'Zone 4':(row.zone3_top,row.upper_band)}
    return mapping.get(zone, (np.nan,np.nan))

def optimise_entry_range(row, direction, preferred_zone):
    low, high = zone_bounds(row, preferred_zone); mid = (low+high)/2 if pd.notna(low) and pd.notna(high) else np.nan
    return {'entry_range_low':min(low,high) if pd.notna(low) else np.nan,'entry_range_high':max(low,high) if pd.notna(high) else np.nan,'entry_mid':mid}

def construct_stop_target(row, direction, entry_mid):
    atr = row.get('atr14', np.nan)
    if pd.isna(atr) or pd.isna(entry_mid) or atr<=0: return {'stop':np.nan,'target':np.nan,'rr':np.nan}
    risk = atr*0.25
    stop = entry_mid-risk if direction=='BUY' else entry_mid+risk
    target = entry_mid+PARAMETERS['minimum_rr']*risk if direction=='BUY' else entry_mid-PARAMETERS['minimum_rr']*risk
    rr = abs(target-entry_mid)/abs(entry_mid-stop) if entry_mid!=stop else np.nan
    return {'stop':stop,'target':target,'rr':rr}

def estimate_trigger_probability(market, direction, zone, trades):
    if trades.empty: return {'trigger_probability':PARAMETERS['fallback_trigger_probability'],'trigger_sample':0,'trigger_source':'fallback'}
    t = add_entry_zone_if_missing(trades)
    exact = t[(t.market.eq(market))&(t.direction.eq(direction))&(t.entry_zone.eq(zone))]
    if len(exact)>=PARAMETERS['min_trigger_sample'] and 'triggered' in exact.columns:
        return {'trigger_probability':float(exact.triggered.astype(bool).mean()),'trigger_sample':int(len(exact)),'trigger_source':'exact_history'}
    market_zone = t[(t.market.eq(market))&(t.entry_zone.eq(zone))]
    if len(market_zone)>=PARAMETERS['min_trigger_sample'] and 'triggered' in market_zone.columns:
        return {'trigger_probability':float(market_zone.triggered.astype(bool).mean()),'trigger_sample':int(len(market_zone)),'trigger_source':'market_zone_history'}
    return {'trigger_probability':PARAMETERS['fallback_trigger_probability'],'trigger_sample':int(len(exact)),'trigger_source':'fallback'}

def calculate_expected_r(template, profile, trigger):
    comps, weights = [], []
    if template.get('template_trades',0)>0: comps.append(float(template.get('template_avg_r',0))); weights.append(min(template.get('template_trades',0),100))
    if profile.get('profile_trades',0)>0: comps.append(float(profile.get('profile_avg_r',0))); weights.append(min(profile.get('profile_trades',0),100))
    raw = float(np.average(comps, weights=weights)) if comps else 0.0
    return {'raw_expected_r':raw,'expected_r':raw*float(trigger.get('trigger_probability', PARAMETERS['fallback_trigger_probability']))}

In [ ]:
# ============================================================
# 7. CONDITION AWARENESS + RECOMMENDATION VERSION ENGINE
# ============================================================
def assess_condition(reco, row):
    current_price, price_at_gen, atr = row.get('current_price',np.nan), reco.get('price_at_generation',np.nan), row.get('atr14',np.nan)
    if pd.isna(current_price) or pd.isna(price_at_gen) or pd.isna(atr) or atr<=0:
        return {'recommendation_validity_status':'DO_NOT_USE_RECALCULATE','requires_refresh':True,'volatility_warning':'Current data incomplete. Refresh before use.','atr_move_since_generation':np.nan}
    atr_move = abs(current_price-price_at_gen)/atr
    if reco.get('zone_at_generation') != row.get('current_zone'):
        return {'recommendation_validity_status':'ZONE_CHANGED','requires_refresh':True,'volatility_warning':'Market zone changed since recommendation generation.','atr_move_since_generation':atr_move}
    if atr_move >= PARAMETERS['force_recalc_atr_threshold']:
        return {'recommendation_validity_status':'DO_NOT_USE_RECALCULATE','requires_refresh':True,'volatility_warning':'Market moved materially. Recalculate before use.','atr_move_since_generation':atr_move}
    if atr_move >= PARAMETERS['stale_atr_threshold']:
        return {'recommendation_validity_status':'STALE_PRICE','requires_refresh':True,'volatility_warning':'Market has moved since generation. Treat levels with caution.','atr_move_since_generation':atr_move}
    return {'recommendation_validity_status':'VALID','requires_refresh':False,'volatility_warning':'','atr_move_since_generation':atr_move}

def build_recommendations(market_state, market_regime, event_risk, templates, profiles, trades, analysts, universe):
    rows=[]; regime_lookup = market_regime.set_index('market').to_dict('index') if not market_regime.empty else {}; universe_lookup = universe.set_index('market').to_dict('index'); risk_group = event_risk.groupby('market') if not event_risk.empty else None
    for _, ms in market_state.iterrows():
        market=ms.market; session=universe_lookup.get(market,{}).get('session','Europe')
        template=select_best_template(market, templates); direction=template['direction']; zone=template['preferred_entry_zone']
        profile=select_best_analyst(market,direction,zone,profiles,analysts,session); entry=optimise_entry_range(ms,direction,zone); stop_target=construct_stop_target(ms,direction,entry['entry_mid']); trigger=estimate_trigger_probability(market,direction,zone,trades); expected=calculate_expected_r(template,profile,trigger)
        if risk_group is not None and market in risk_group.groups:
            top = risk_group.get_group(market).sort_values('risk_score', ascending=False).iloc[0]; event_status=top.event_risk_status; event_warning=top.analyst_warning
        else: event_status='NONE'; event_warning=''
        regime=regime_lookup.get(market,{})
        reco = {'recommendation_version_id':str(uuid.uuid4()),'opportunity_id':f'{pd.Timestamp.utcnow().date()}_{market}_{session}_v1','market':market,'session':session,'generated_at':pd.Timestamp.utcnow(),'version_number':1,'direction':direction,'preferred_entry_zone':zone,'current_zone':ms.current_zone,'analyst_action':'ENTER_NOW' if ms.current_zone==zone else 'WAIT_FOR_PREFERRED_ZONE','assigned_analyst':profile.get('assigned_analyst'),'eligible_analysts':profile.get('eligible_analysts'),'entry_range_low':entry['entry_range_low'],'entry_range_high':entry['entry_range_high'],'entry_mid':entry['entry_mid'],'stop':stop_target['stop'],'target':stop_target['target'],'rr':stop_target['rr'],'trigger_probability':trigger['trigger_probability'],'expected_r':expected['expected_r'],'raw_expected_r':expected['raw_expected_r'],'price_at_generation':ms.current_price,'zone_at_generation':ms.current_zone,'event_risk_status':event_status,'event_warning':event_warning,'regime_tags':regime.get('regime_tags',[]),'trend_state':regime.get('trend_state'),'volatility_state':regime.get('volatility_state'),**template,**profile,'parameter_snapshot':PARAMETER_SNAPSHOT,'parameter_snapshot_hash':PARAMETER_SNAPSHOT_HASH}
        reco.update(assess_condition(reco, ms)); rows.append(reco)
    return pd.DataFrame(rows)

recommendations = build_recommendations(market_state, market_regime, event_risk, template_profiles, analyst_profiles, historical_trades, ACTIVE_ANALYSTS, MARKET_UNIVERSE)
display(recommendations.head())

In [ ]:
# ============================================================
# 8. SHADOW ENGINE + COACHING ENGINE
# ============================================================
def build_shadow_trades(recommendations):
    if recommendations.empty: return pd.DataFrame()
    shadow = recommendations.copy(); shadow['shadow_trade_id']=[str(uuid.uuid4()) for _ in range(len(shadow))]; shadow['entry']=shadow.entry_mid; shadow['visible_to_analyst']=False; shadow['shadow_status']=np.where(shadow.recommendation_validity_status.eq('VALID'),'ACTIVE','WATCH'); shadow['created_at']=pd.Timestamp.utcnow()
    keep=['shadow_trade_id','recommendation_version_id','opportunity_id','market','session','direction','entry','stop','target','rr','trigger_probability','expected_r','shadow_status','created_at','visible_to_analyst','parameter_snapshot_hash']
    return shadow[keep]

def lint_analyst_text(text):
    low=text.lower(); hits=[term for term in PARAMETERS['forbidden_analyst_terms'] if term.lower() in low]; return len(hits)==0, hits

def generate_coaching_note(row):
    text = f"{row.market} is currently in {row.current_zone}. The historical setup profile favours {row.direction} interest around {row.preferred_entry_zone}. The suggested entry region is {row.entry_range_low:.5g} to {row.entry_range_high:.5g}, with an expected trigger probability of {row.trigger_probability:.0%} and expected opportunity of {row.expected_r:.2f}R."
    if row.recommendation_validity_status!='VALID' and row.volatility_warning: text += f" Current condition note: {row.volatility_warning}"
    if row.event_warning: text += f" Event risk note: {row.event_warning}"
    text += ' Treat this as a coaching range rather than an instruction; execution judgement remains important.'
    ok, hits = lint_analyst_text(text)
    return text if ok else 'This market has a historically relevant setup today. Review the suggested range, risk area and target area with current conditions in mind.'

def build_coaching_recommendations(recommendations):
    if recommendations.empty: return pd.DataFrame()
    out = recommendations.copy(); out['coaching_recommendation_id']=[str(uuid.uuid4()) for _ in range(len(out))]
    out['suggested_entry_range']=out.apply(lambda r: f"{r.entry_range_low:.5g} - {r.entry_range_high:.5g}", axis=1)
    out['suggested_risk_range']=out.apply(lambda r: f"Around {r.stop:.5g}", axis=1); out['suggested_target_range']=out.apply(lambda r: f"Towards {r.target:.5g}", axis=1)
    out['coaching_note']=out.apply(generate_coaching_note, axis=1); out['shown_at']=pd.Timestamp.utcnow()
    keep=['coaching_recommendation_id','recommendation_version_id','opportunity_id','market','session','assigned_analyst','direction','current_zone','preferred_entry_zone','analyst_action','suggested_entry_range','suggested_risk_range','suggested_target_range','trigger_probability','expected_r','event_risk_status','recommendation_validity_status','coaching_note','shown_at']
    return out[keep]

shadow_trades = build_shadow_trades(recommendations)
coaching_recommendations = build_coaching_recommendations(recommendations)
display(shadow_trades.head()); display(coaching_recommendations.head())

In [ ]:
# ============================================================
# 9. REVIEW ENGINE + ALLOCATION ENGINE
# ============================================================
def build_post_trade_reviews(actual_trades, recommendations):
    if actual_trades.empty or recommendations.empty: return pd.DataFrame()
    t=actual_trades.copy()
    if 'historical_backfill' in t.columns: t=t[t.historical_backfill.astype(bool)==False].copy()
    if t.empty: return pd.DataFrame()
    merged=t.merge(recommendations, on=['market','session'], how='left', suffixes=('_actual','_reco'))
    merged['direction_alignment']=np.where(merged.get('direction_actual', merged.get('direction')).eq(merged.get('direction_reco', merged.get('direction'))), 'Aligned', 'Different')
    merged['entry_alignment']=merged.apply(lambda r: 'High' if pd.notna(r.get('entry')) and pd.notna(r.get('entry_range_low')) and r.entry_range_low <= r.entry <= r.entry_range_high else 'Low', axis=1)
    merged['alignment_score']=(merged.direction_alignment.eq('Aligned')).astype(int)+(merged.entry_alignment.eq('High')).astype(int)
    merged['analyst_facing_review']='Review generated against the recommendation shown. Use this as a learning point rather than a judgement.'
    merged['review_id']=[str(uuid.uuid4()) for _ in range(len(merged))]
    return merged

def allocate_coverage(recommendations, active_analysts):
    if recommendations.empty: return pd.DataFrame()
    workload={a:0 for a in active_analysts[active_analysts.active].analyst.tolist()}; rows=[]
    for _, r in recommendations.sort_values('expected_r', ascending=False).iterrows():
        eligible = r.eligible_analysts if isinstance(r.eligible_analysts, list) and r.eligible_analysts else list(workload.keys())
        scores=[]
        for a in eligible:
            score=float(r.expected_r)+(0.2 if a==r.assigned_analyst else 0)-workload.get(a,0)*0.1; scores.append((a,score))
        assigned=sorted(scores, key=lambda x:x[1], reverse=True)[0][0]; workload[assigned]=workload.get(assigned,0)+1
        rows.append({'allocation_id':str(uuid.uuid4()),'opportunity_id':r.opportunity_id,'recommendation_version_id':r.recommendation_version_id,'market':r.market,'session':r.session,'assigned_analyst':assigned,'eligible_analysts':eligible,'allocation_score':sorted(scores, key=lambda x:x[1], reverse=True)[0][1],'allocation_status':'ASSIGNED','reason_summary':'Assigned using expected R, profile fit and workload balancing.'})
    return pd.DataFrame(rows)

post_trade_reviews = build_post_trade_reviews(historical_trades, recommendations)
coverage_allocation = allocate_coverage(recommendations, ACTIVE_ANALYSTS)
display(coverage_allocation.head()); display(coverage_allocation.assigned_analyst.value_counts())

In [ ]:
# ============================================================
# 10. EXPORTS + VALIDATION SUMMARY
# ============================================================
EXPORT_DIR = '/mnt/data/apip_research_engine_exports'
os.makedirs(EXPORT_DIR, exist_ok=True)
exports = {'market_state':market_state,'market_regime':market_regime,'event_risk':event_risk,'template_profiles':template_profiles,'analyst_profiles':analyst_profiles,'recommendations':recommendations,'shadow_trades_internal':shadow_trades,'coaching_recommendations':coaching_recommendations,'coverage_allocation':coverage_allocation,'post_trade_reviews':post_trade_reviews}
manifest={'run_id':RUN_ID,'run_started_at':RUN_STARTED_AT,'parameter_snapshot_hash':PARAMETER_SNAPSHOT_HASH,'exports':{}}
for name, df in exports.items():
    path=os.path.join(EXPORT_DIR, f'{name}.csv'); df.to_csv(path, index=False); manifest['exports'][name]={'path':path,'rows':len(df),'columns':list(df.columns)}
with open(os.path.join(EXPORT_DIR, 'manifest.json'), 'w') as f: json.dump(manifest, f, indent=2, default=str)
summary = pd.DataFrame([{'metric':'recommendations','value':len(recommendations)},{'metric':'enter_now','value':int((recommendations.analyst_action=='ENTER_NOW').sum())},{'metric':'wait','value':int((recommendations.analyst_action=='WAIT_FOR_PREFERRED_ZONE').sum())},{'metric':'avg_expected_r','value':float(recommendations.expected_r.mean())},{'metric':'avg_trigger_probability','value':float(recommendations.trigger_probability.mean())},{'metric':'requires_refresh','value':int(recommendations.requires_refresh.astype(bool).sum())}])
display(summary)
print('Exports written to:', EXPORT_DIR)

# Production translation note

Claude should treat this notebook as a behavioural reference implementation, not production code to port line-by-line.

For the same inputs and parameter snapshot:

```text
Research Notebook Recommendation ≈ Production Platform Recommendation
```

The implementation can differ. The behaviour should not.